In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

from config import DATA_DIR, CHROMA_DIR, CHUNK_SIZE, CHUNK_OVERLAP, TOP_K_RETRIEVAL
print('Config loaded ✅')
print(f'DATA_DIR   : {DATA_DIR}')
print(f'CHROMA_DIR : {CHROMA_DIR}')

In [ ]:
from rag.loader import load_pdfs

# Change this path to any PDF you want to test
PDF_DIR = DATA_DIR

documents = load_pdfs(PDF_DIR)
print(f'\nLoaded {len(documents)} pages')

# Preview first page
if documents:
    first = documents[0]
    print(f"\n--- First page preview ---")
    print(f"Source : {first['source']}")
    print(f"Page   : {first['page']}")
    print(f"Text   : {first['text'][:500]}...")

In [ ]:
from rag.chunker import chunk_documents

chunks = chunk_documents(documents, chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)

print(f'\nTotal chunks created : {len(chunks)}')
print(f'Chunk size           : {CHUNK_SIZE} chars')
print(f'Overlap              : {CHUNK_OVERLAP} chars')

# Show chunk distribution per source
from collections import Counter
source_counts = Counter(c['source'] for c in chunks)
print('\nChunks per file:')
for src, count in source_counts.items():
    print(f'  {src}: {count} chunks')

In [ ]:
# Preview a specific chunk
chunk_index = 0  # ← Change this to inspect different chunks
c = chunks[chunk_index]
print(f"Chunk ID : {c['chunk_id']}")
print(f"Source   : {c['source']}  |  Page: {c['page']}")
print(f"Length   : {len(c['text'])} chars")
print("\n" + "-"*50)
print(c['text'])

In [ ]:
from rag.embedder import get_embedding_function

embed_fn = get_embedding_function()

# Embed a single sample text
sample_text = chunks[0]['text'] if chunks else 'Pradita University offers quality education.'
vector = embed_fn.embed_query(sample_text)

print(f'Embedding dimensions : {len(vector)}')
print(f'First 10 values      : {[round(v, 4) for v in vector[:10]]}')

In [ ]:
from rag.store import get_chroma_collection, add_documents_to_db

collection = get_chroma_collection(CHROMA_DIR)
print(f'Docs before insertion : {collection.count()}')

add_documents_to_db(chunks, collection)
print(f'Docs after  insertion : {collection.count()}')

In [ ]:
from rag.retriever import retrieve_context

# ← Change this query to test different questions
QUERY = 'What programs does Pradita University offer?'

result = retrieve_context(QUERY, top_k=TOP_K_RETRIEVAL, collection=collection)

print(f'Query    : {QUERY}')
print(f'Sources  : {result["sources"]}')
print(f'Chunks retrieved : {len(result["chunks"])}')
print('\n' + '='*60 + '\nFULL CONTEXT:\n' + '='*60)
print(result['context'])

In [ ]:
# Show each chunk with distance score
print('Chunk Details:')
for i, chunk in enumerate(result['chunks'], 1):
    print(f"\n[Chunk {i}] distance={chunk['distance']:.4f} | {chunk['source']} p.{chunk['page']}")
    print(chunk['text'][:200] + '...' if len(chunk['text']) > 200 else chunk['text'])